# 動画をそのまま ER 2 に渡してシャトルを追う — Gemini Robotics ER 2

`er2_video_shuttle.ipynb` はコマを切り出して 1 枚ずつ聞きました。こちらは **動画ファイルをそのまま ER 2 に渡し**、
「何秒にシャトルがどこにあるか」を **1 回の呼び出し**で答えさせます。

- 呼び出しは 1 回（コマ割り版は `DURATION × SAMPLE_FPS` 回）
- 動画の長さぶんトークンを使う（目安: 1 秒 ≒ 300 トークン前後。10 秒なら数円）
- 位置の精度はコマ割り版より落ちることがある。返った時刻のコマに点を描いて確かめる

前提: 左の 🔑（シークレット）に `GEMINI_API_KEY` を登録し、「ノートブックからのアクセス」を ON にしておく。


## 1. セットアップ

In [ ]:
%pip install -U -q "google-genai>=2.9.0" pydantic japanize-matplotlib

In [ ]:
from google.colab import userdata
from google import genai

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("ERROR: 左の 🔑 に GEMINI_API_KEY を登録し、ノートブックからのアクセスを ON にしてください")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-robotics-er-2-preview"

response = client.interactions.create(model=MODEL_ID, input="Hello Physical World?")
print(response.output_text)


In [ ]:
import json
import os
import time
from typing import List, Optional

import cv2
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
from PIL import Image, ImageDraw
from pydantic import BaseModel, Field
from IPython.display import HTML, display


## 2. 動画を Google Drive から読み、渡す区間を切り出す

長い動画はトークンを使うので、まず **数秒〜十数秒** に切って渡します（ffmpeg で切り出し。再エンコードなしなので数秒で終わります）。
`START_SEC` と `DURATION` を変えて区間を選んでください。全部渡すなら `DURATION = None`。


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

VIDEO_PATH = '/content/drive/MyDrive/badminton.mp4'   # ← 自分の動画に変える
START_SEC  = 0.0
DURATION   = 8.0     # 秒。None なら全部

assert os.path.exists(VIDEO_PATH), f"見つかりません: {VIDEO_PATH}"

CLIP_PATH = "/content/clip.mp4"
if DURATION is None:
    CLIP_PATH = VIDEO_PATH
else:
    !ffmpeg -y -loglevel error -ss {START_SEC} -t {DURATION} -i "{VIDEO_PATH}" -c copy {CLIP_PATH}

cap = cv2.VideoCapture(CLIP_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
clip_len = n_total / fps
print(f"渡す動画: {CLIP_PATH}  {W}x{H}, {fps:.1f} fps, {clip_len:.1f} 秒, {os.path.getsize(CLIP_PATH)/1e6:.1f} MB")


## 3. Files API にアップロードする

動画は base64 で埋め込まず、いちど Files API に置いて URI で渡します（48 時間で自動削除）。
処理が終わるまで数十秒待ちます。


In [ ]:
print("動画をアップロードしています...")
vid = client.files.upload(file=CLIP_PATH)
while vid.state.name == "PROCESSING":
    print(".", end="", flush=True)
    time.sleep(2)
    vid = client.files.get(name=vid.name)
print("\n準備完了:", vid.uri, vid.state.name)


## 4. 「何秒にシャトルはどこか」を 1 回で聞く

時刻と位置の組を列挙させます。**見えない時刻は含めなくてよい**と明記し、個数は指定しません
（「N 個返して」と頼むと、無いところに点を置きます）。


In [ ]:
class ShuttleAt(BaseModel):
    second: float = Field(description="動画の先頭からの時刻（秒・小数可）")
    point: List[int] = Field(description="その時刻のシャトルの位置 [y, x]（0〜1000 に正規化）")
    note: str = Field(description="そのときの状況を短く（日本語）。例: 手前の選手が打った直後")


class ShuttleTrack(BaseModel):
    items: List[ShuttleAt] = Field(description="シャトルが見えている時刻と位置。見えない時刻は含めない。無ければ空のリスト")
    summary: str = Field(description="ラリーの流れの要約（日本語・2文まで）")


PROMPT = (
    "バドミントンの動画です。シャトル（羽根）が見えている時刻を約0.5秒おきに選び、"
    "それぞれの時刻でのシャトルの位置を [y, x]（0〜1000）で返してください。"
    "見えない・判別できない時刻は含めないでください。個数は指定しません。"
    "ラケットや白い線をシャトルと見間違えないこと。"
)

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "video", "uri": vid.uri},
        {"type": "text", "text": PROMPT},
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={"type": "text", "mime_type": "application/json",
                     "schema": ShuttleTrack.model_json_schema()},
)

track = ShuttleTrack.model_validate_json(response.output_text)
print("要約:", track.summary)
print(f"{len(track.items)} 点")
for it in track.items:
    print(f"{it.second:6.2f}s  point={it.point}  {it.note}")


## 5. 返った時刻のコマに点を描いて確かめる

ER 2 が言った時刻のコマを切り出し、その上に返った点を描きます。
点がシャトルに乗っていなければ、時刻がずれているのか、位置がずれているのかをコマで見分けます。


In [ ]:
def frame_at(path, sec):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(round(sec * fps)))
    ok, bgr = cap.read()
    cap.release()
    return Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)) if ok else None


def draw(img, point, label):
    out = img.copy()
    d = ImageDraw.Draw(out)
    W_, H_ = out.size
    y, x = point
    px, py = x / 1000 * W_, y / 1000 * H_
    r = max(8, W_ // 80)
    d.ellipse([px - r, py - r, px + r, py + r], outline=(255, 40, 40), width=max(3, W_ // 300))
    d.line([px - 2 * r, py, px + 2 * r, py], fill=(255, 40, 40), width=2)
    d.line([px, py - 2 * r, px, py + 2 * r], fill=(255, 40, 40), width=2)
    d.rectangle([0, 0, 200, 34], fill=(0, 0, 0))
    d.text((8, 8), label, fill=(255, 255, 255))
    return out


annotated = []
for it in track.items:
    img = frame_at(CLIP_PATH, min(it.second, clip_len - 1 / fps))
    if img is not None:
        annotated.append(draw(img, it.point, f"{it.second:.2f}s"))

if not annotated:
    print("描けるコマがありません（items が空か、時刻が動画の範囲外）")
else:
    cols = min(4, len(annotated))
    rows = (len(annotated) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 2.4 * rows))
    for ax, im in zip(np.array(axes).ravel(), annotated):
        ax.imshow(im); ax.axis("off")
    for ax in np.array(axes).ravel()[len(annotated):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()


## 6. 軌跡を 1 枚に重ねる

返った点を時刻順につないで、最初のコマの上に描きます。シャトルの飛び方（山なり・直線）が見えるかを確かめます。


In [ ]:
if track.items:
    base = frame_at(CLIP_PATH, track.items[0].second)
    d = ImageDraw.Draw(base)
    pts = [(it.point[1] / 1000 * base.size[0], it.point[0] / 1000 * base.size[1]) for it in sorted(track.items, key=lambda i: i.second)]
    for a, b in zip(pts, pts[1:]):
        d.line([a, b], fill=(255, 220, 0), width=max(3, base.size[0] // 300))
    for (px, py), it in zip(pts, sorted(track.items, key=lambda i: i.second)):
        r = max(6, base.size[0] // 120)
        d.ellipse([px - r, py - r, px + r, py + r], fill=(255, 40, 40))
        d.text((px + r + 2, py - r), f"{it.second:.1f}", fill=(255, 255, 255))
    plt.figure(figsize=(10, 6)); plt.imshow(base); plt.axis("off"); plt.title("ER 2 が返した位置を時刻順につないだもの"); plt.show()


## コマ割り版との違い

| | この版（動画をそのまま渡す） | `er2_video_shuttle.ipynb`（コマ割り） |
|---|---|---|
| 呼び出し回数 | 1 回 | コマ数ぶん |
| 時刻の刻み | ER 2 が選ぶ（約 0.5 秒おきと頼んでいる） | 自分で決める（`SAMPLE_FPS`） |
| 位置の精度 | 動画全体を見て答えるので粗くなりがち | 1 枚を見るので高い |
| 向いていること | ラリーの流れ・打球の時刻・どこで何が起きたか | 特定のコマでの正確な位置 |

両方を同じ区間で走らせて、点の位置がどれくらい違うかを見ると、動画理解と画像理解の違いが分かります。
